# Interactive IQ fitting across a series with `pipeline_v2`

This notebook shows how to use `citkid.pipeline_v2.interactive.run_iq_series` to fit IQ loops across a series of measurements, such as a series in power, temperature, or field.

Each series point gets its own `DataSet` and `AnalysisRunner`, but they all share one interactive window. The window lets you move through resonators and series points, rerun panels, save results, and mark data as bad.

---

## What `run_iq_series` expects

`run_iq_series(...)` needs:

1. A function `make_custom_steps(series_idx)` that returns the calibration loading steps for one series index.
2. A calibration YAML and analysis YAML that describe the IQ analysis pipeline.
3. An output zarr root where each series point is stored in its own subgroup.
4. A per-resonator x value and y value for the left-hand series scatter plot.

`pipeline_v2` safety rules still apply inside the IQ series window:

- Running one panel invalidates later panels for the current `(series_idx, data_idx)` selection.
- Later panels are cleared and marked as needing a rerun rather than being silently recomputed.
- The UI asks before you save or navigate away while downstream panels still need a rerun.
- Leading global/global-res analysis steps are only run automatically when their outputs do not already exist for that series runner.

In [ ]:
import numpy as np
import zarr
from citkid.pipeline_v2.framework import plStep
from citkid.pipeline_v2.interactive import run_iq_series

# ---------------------------------------------------------------------
# Raw input and output stores
# ---------------------------------------------------------------------

# Raw data source (read-only)
root_raw = zarr.open(
    'path/to/raw_data.zarr',
    mode='r',
)

# Output root. iq_series creates/uses subgroups like
# series_000, series_001, ... inside this store.
root_out = zarr.open(
    'path/to/output_iq_series.zarr',
    mode='a',
)

# Number of series points and resonators
n_series = ...   # int
nrows   = ...   # int

## Define the per-series calibration loading steps

`make_custom_steps(series_idx)` must return the calibration steps for exactly one series point. These steps usually load:

- one shared global block (`nrows`, `fres_all`, `qres_all`, ...)
- one global-res block (`fres`, `qres`, `ares`, `res_idxs`, ...)
- per-row fine sweep data (`ff`, `zf`)
- per-row gain sweep data (`fg`, `zg`)

If a value should differ by resonator, make it `per-row` or `vectorized`, not `global`.

In [ ]:
def make_custom_steps(series_idx):
    """Return the calibration-loading steps for one series index."""

    def load_global_data():
        # Example: values shared across all resonators for this series index
        fres_all = ...
        qres_all = ...
        return fres_all, qres_all, nrows

    def load_global_res_data():
        # Example: one value per resonator, computed once for all rows
        fres = ...
        qres = ...
        ares = ...      # often the x-axis value for the series plot
        res_idxs = ...
        return fres, qres, ares, res_idxs

    def load_data_f(data_idx):
        # Fine sweep for one resonator. Frequencies should be ascending.
        ff = ...
        zf = ...
        return ff, zf

    def load_data_g(data_idx):
        # Gain sweep for one resonator. Frequencies should be ascending.
        fg = ...
        zg = ...
        return fg, zg

    custom_steps = [
        plStep('load_global_data', load_global_data, [], ['fres_all', 'qres_all', 'nrows'], 'global'),
        plStep('load_global_res_data', load_global_res_data, [], ['fres', 'qres', 'ares', 'res_idxs'], 'global-res'),
        plStep('load_data_f', load_data_f, ['data_idx'], ['ff', 'zf'], 'per-row'),
        plStep('load_data_g', load_data_g, ['data_idx'], ['fg', 'zg'], 'per-row'),
    ]
    return custom_steps

## Choose the scatter-plot axes

The series window needs one x value and one y value for each `(series_idx, data_idx)` pair.

- `x_param_name` is a `DataSet` attribute loaded from the runner, usually something like `ares` or `fres`.
- `y_param_name` is the IQ fit parameter on the y-axis: one of `'fr'`, `'Qr'`, `'amp'`, `'phi'`, `'a'`, `'Qc'`, `'Qi'`.

Pass `data_idxs` to review only a subset of resonators. Navigation and background fitting then only visit those rows, and `start_idx` is a position in `data_idxs` (not a `data_idx`). Leave `start_idx = None` to resume where you left off (see below).

## Optional: fit y vs x for each resonator

Pass a `SeriesXYFit` to fit the series plot's y vs x data, one fit per resonator. `fit(x, y)` gets exactly the plotted points that have both x and y (points marked bad are left out), sorted by x, and returns its outputs in `output_names` order. `model(xs, *outputs)` draws the fitted curve. If you want a log scale, take the log inside `fit` and `model`.

- Each resonator is fitted in the background as soon as all its series points are pre-fitted, so resonators you never open still get a fit.
- When you change a resonator's data (rerun a panel, mark points bad, apply to all), it is refitted. Opening a resonator without changing anything doesn't refit it.
- The outputs are shown in the series plot title and saved to `root_out['xy_fit']` (or `group=` if you pass one): one array per output, plus `fit_x` / `fit_y` with the inputs of each fit.

In [ ]:
from citkid.pipeline_v2.interactive import SeriesXYFit

def fit_line(x, y):
    slope, intercept = np.polyfit(x, y, 1)
    return slope, intercept

xy_fit = SeriesXYFit(
    fit = fit_line,
    output_names = ['slope', 'intercept'],
    model = lambda xs, slope, intercept: slope * xs + intercept,
    name = 'a_vs_power',
)

### Optional: use DataSets you already have

Instead of `make_custom_steps`, you can pass `datasets`: one existing `DataSet` per series point, in order, all with the same number of rows. Their stored results are used and extended, wherever they live. In that case `n_series` is `len(datasets)`, and `root` is optional: it only holds the session state (for resuming) and the xy fits, so leave it out if you don't need to resume.

The background worker opens its own copies of the datasets (`DataSet.copy()`), so they must be reopenable: either built with `custom_path` / an embedded definition, or with `custom_cal_steps` in this session. While the window is open, every dataset writes through its fast write buffer; the buffer is merged and each dataset's own `write_buffer` setting is restored when the window closes.

If the DataSets don't store the x quantity but each one has a single known value (e.g. each DataSet is one drive power), pass `x_values=[...]`, one number per series point, in place of `x_param_name`, and label it with `x_name`.

In [ ]:
from citkid.pipeline_v2 import DataSet

# For example, one DataSet per series point in separate zarr files:
# datasets = [
#     DataSet(zarr_path=f'path/to/point_{i}.zarr', cal_yaml_path='iq', custom_path='custom_steps.py')
#     for i in range(n_series)
# ]
# run_iq_series(
#     datasets = datasets,
#     root = zarr.open_group('path/to/series_state.zarr', mode='a'),   # optional
#     ...,                                                              # other arguments as below
# )

In [ ]:
run_iq_series(
    make_custom_steps = make_custom_steps,
    cal_yaml_path = 'iq',
    analysis_yaml_path = 'iq',
    root = root_out,
    n_series = n_series,
    x_param_name = 'ares',   # DS attribute used on the x-axis
    x_name = 'Power (uW)',
    y_param_name = 'a',      # IQ fit parameter on the y-axis
    start_series_idx = 0,
    start_idx = None,        # None = first data_idx not yet viewed; or a position in data_idxs
    # data_idxs = [0, 3, 7], # optional subset of rows; None = all rows
    # xy_fit = xy_fit,       # optional y vs x fit per resonator (see above)
    # x_values = [...],      # one fixed x per series point, instead of x_param_name
)

## Practical notes

- The IQ series window saves panel outputs when you explicitly save, when you change selection, or when the window closes.
- Re-running an earlier panel marks later panels stale for the current selection. The window clears those plots and asks before you leave that stale state.
- Pre-fitting runs in the background: every resonator in `data_idxs` is fitted for every series point, starting after the current one. A resonator whose fit failed is not retried automatically; rerun its panels by hand.
- Background prefetch is read-only. It prepares plot caches for the next resonator but does not execute pipeline steps or save anything.
- Leading global/global-res analysis steps are initialized once per series runner when needed. If they already exist in zarr, the window loads them instead of rerunning them.
- If you need to deliberately rerun a global/global-res analysis step after outputs already exist, do that from code with `allow_global_step_overwrite=True`; the UI is intentionally conservative by default.

## Stopping and resuming

You can close the window partway through and pick up later by running the same cell again with the same `root`.

- **Finish checking the open resonator before you close the window.** A `data_idx` counts as *viewed* when you move to another resonator or close the window while it is open, so the resonator on screen when you close is recorded as done.
- With `start_idx = None`, the window opens at the first entry of `data_idxs` you haven't viewed. For example, if you reviewed `data_idxs = [0, 1, 3, 4, 5]` and closed on 3, a later run over all resonators opens at 2.
- Fits saved in earlier sessions are loaded, not rerun, and resonators that were already pre-fitted (including failed fits) are skipped.
- The progress is stored in `root.attrs['series_state']`. To start over, delete it: `del root_out.attrs['series_state']`.

### Storage

During a session, results are written to a fast buffer with one small file per resonator. When you close the window it waits for the current background fit, then merges the buffer into the compact (sharded) arrays, which can take a few seconds. Close the window normally before copying the zarr to another computer. If Python crashes before closing, nothing is lost: the data is still readable, and it is merged the next time you close the IQ series window. You can also merge it yourself with `DS.consolidate_storage()` on each series point's `DataSet`.